In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import mlflow
import optuna
from optuna.integration import MLflowCallback
from sqlalchemy import create_engine
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss
from collections import defaultdict

TABLE_NAME = 'clean_users_churn'

TRACKING_SERVER_HOST = '127.0.0.1'
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'churn_laptev_optuna_2'
RUN_NAME = "model_bayesian_search"

STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "churn_model_12"

In [7]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = 'https://storage.yandexcloud.net'
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')


mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [8]:
# Загрузка данных
DB_DESTINATION_USER = os.environ["DB_DESTINATION_USER"]
DB_DESTINATION_PASSWORD = os.environ["DB_DESTINATION_PASSWORD"]
DB_DESTINATION_HOST = os.environ["DB_DESTINATION_HOST"]
DB_DESTINATION_PORT = os.environ["DB_DESTINATION_PORT"]
DB_DESTINATION_NAME = os.environ["DB_DESTINATION_NAME"]


conn = create_engine(f"postgresql://{DB_DESTINATION_USER}:{DB_DESTINATION_PASSWORD}@{DB_DESTINATION_HOST}:{DB_DESTINATION_PORT}/{DB_DESTINATION_NAME}")
df = pd.read_sql(f"select * from {TABLE_NAME}", conn)

In [9]:
features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = 'begin_date'
stratify_column = 'target'
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(df[features],
                                                    df[target],
                                                    test_size=test_size,
                                                    shuffle=False)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

Размер выборки для обучения: (5615, 3)
Размер выборки для теста: (1404, 3)


In [10]:
def objective(trial: optuna.Trial) -> float:
    param = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "depth": trial.suggest_int("depth", 1, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5),
        "loss_function": "Logloss",
        "task_type": "CPU",
        "random_seed": 0,
        "iterations": 300,
        "verbose": False,
    }

    model = CatBoostClassifier(**param)
    skf = StratifiedKFold(n_splits=2)
    metrics = defaultdict(list)
    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        train_x = X_train.iloc[train_index]
        train_y = y_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        val_y = y_train.iloc[val_index]
        
        # Обучение модели
        model.fit(train_x, train_y)
    
        # Предсказания
        prediction = model.predict(val_x)
        probas = model.predict_proba(val_x)[:, 1]

        _, err1, _, err2 = confusion_matrix(val_y, prediction, normalize='all').ravel()
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, prediction)
        
        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)


    # ваш код здесь #
    err_1 = np.median(np.array(metrics['err1']))
    err_2 = np.median(np.array(metrics['err2']))
    auc = np.median(np.array(metrics['auc']))
    precision = np.median(np.array(metrics['precision']))
    recall = np.median(np.array(metrics['recall']))
    f1 = np.median(np.array(metrics['f1']))
    logloss = np.median(np.array(metrics['logloss']))
		
    return auc

In [11]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id
    

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

mlflc = MLflowCallback(tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
                       metric_name='AUC',
                       create_experiment=False,
                       mlflow_kwargs={'experiment_id': experiment_id, 'tags': {'mlflow.parentRunId': run_id}})

study = optuna.create_study(sampler=optuna.samplers.TPESampler(),
                            direction='maximize',
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME)

study.optimize(objective, n_trials=10, callbacks=[mlflc])
best_params = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params}")
print(f'RUN ID for {RUN_NAME} is {run_id}')

[I 2024-11-11 13:05:30,050] A new study created in RDB with name: churn_model_12
[I 2024-11-11 13:05:32,396] Trial 0 finished with value: 0.7727193824390686 and parameters: {'learning_rate': 0.012302005013247689, 'depth': 6, 'l2_leaf_reg': 1.634058423924152, 'random_strength': 0.522577009430986}. Best is trial 0 with value: 0.7727193824390686.
[I 2024-11-11 13:05:33,896] Trial 1 finished with value: 0.7613110050536229 and parameters: {'learning_rate': 0.0042204994353805935, 'depth': 7, 'l2_leaf_reg': 0.5448528441740618, 'random_strength': 3.938346984664712}. Best is trial 0 with value: 0.7727193824390686.
[I 2024-11-11 13:05:42,169] Trial 2 finished with value: 0.7449550963448841 and parameters: {'learning_rate': 0.0792888497458084, 'depth': 11, 'l2_leaf_reg': 1.5090142235564168, 'random_strength': 0.18862739032821849}. Best is trial 0 with value: 0.7727193824390686.
[I 2024-11-11 13:05:43,002] Trial 3 finished with value: 0.7270261876732682 and parameters: {'learning_rate': 0.00249474

Number of finished trials: 10
Best params: {'learning_rate': 0.04159347338891062, 'depth': 5, 'l2_leaf_reg': 4.361575895608305, 'random_strength': 4.27735999580617}
RUN ID for model_bayesian_search is fcadcf5f279a4b7ca938d40eff904875


In [16]:
best_model = CatBoostClassifier(**best_params, 
                                loss_function='Logloss',
                                task_type='CPU',
                                random_seed=0,
                                iterations=300,
                                verbose=False)
best_model.fit(X_train, y_train)

prediction = best_model.predict(X_test)
probas = best_model.predict_proba(X_test)[:, 1]

In [17]:
# расчёт метрик качества
metrics = {}

_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [19]:
# настройки для логирования в MLFlow
pip_requirements = '../requirements.txt'
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_id='fcadcf5f279a4b7ca938d40eff904875', experiment_id=experiment_id, run_name='Ustal_optuna_logging_best_catboost') as run:
    run_id = run.info.run_id
    
    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    cv_info = mlflow.sklearn.log_model(best_model, artifact_path='cv')
    model_info = mlflow.catboost.log_model(artifact_path='models',
                                           cb_model=best_model,
                                           signature=signature,
                                           input_example=input_example,
                                           registered_model_name='cb_with_optuna_1',
                                           await_registration_for=60,
                                           pip_requirements=pip_requirements)
    
    print(f'RUN_ID is {run_id}')

Successfully registered model 'cb_with_optuna_1'.
2024/11/11 13:15:47 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: cb_with_optuna_1, version 1


RUN_ID is fcadcf5f279a4b7ca938d40eff904875


Created version '1' of model 'cb_with_optuna_1'.
